# Page 5 — "How to Reason": the prompt ladder

This notebook is the empirical backing for **page 5** of the deck, and it is meant to be run,
not just read. Everything the page claims is produced here; the page renders committed logs and
runs no model of its own.

**The claim the page makes.** Changing the *words in the prompt* changes the model's behaviour —
from a one-line blurt to a worked chain — and that change is worth measuring rather than
asserting.

**The three beats, which this notebook follows in order:**

1. **One line of prompt, two behaviours.** Same model, same question, same decoding. Add four
   words and a chain appears. Both traces are real. → *Figure 1*
2. **So what exactly do you say?** Five prompt formulations, measured on the same items. The
   ranking is a real question, not a foregone conclusion. → *Figure 2*
3. **The chain costs something.** The formulation that buys the most accuracy also writes the
   most. → *the close, and the handoff to page 6*

---

## The one constraint that makes this a demonstration rather than a product tour

**Thinking is OFF for every generation in this notebook.**

Qwen3 is a hybrid model: it has a switch that turns an internal reasoning block on and off. If
we used that switch, we would be demonstrating a *feature*, not a *technique* — of course the
model reasons when you press the reason button. So the switch stays off throughout, the weights
never move, the decoding never changes, and **the only variable on this page is the text of the
prompt.**

Section 4 below shows the rendered prompt with the switch off, so you can see for yourself that
we never flipped it. A skeptical reader should check that cell first.

---

## What is inherited and what is ours

| Piece | Source | Why |
|---|---|---|
| `load_math500_test`, `render_prompt`, `extract_final_candidate`, `grade_answer` | `reasoning-from-scratch` ch03 | Grading is not ours to defend. Text in, text out, no model dependency. |
| `get_device` | ch02 | One device-selection function, not two. |
| The zero-shot CoT mechanism (`prompt + "\n\nExplain step by step."`) | ch04 §4.3 | One line. We take the mechanism, not the book's numbers. |
| The generation loop | plain 🤗 `transformers` | Model-agnostic, so swapping checkpoints is cheap. |
| The prompt ladder, the sweep, the exemplar harvest, every number on the page | **ours** | This is the experiment. |

The techniques are cited on the page (Wei et al. 2022 for few-shot chain-of-thought, Kojima et
al. 2022 for the zero-shot phrase). **The measurements are ours** — no citation on this page
carries one of our numbers.

---

## What this notebook writes

Into `out/page_05/`:

| File | Contents | Ships with the page? |
|---|---|---|
| `traces.jsonl` | one line per generation: full trace, token strings, extraction, correctness | linked as a download |
| `derived.json` | the aggregates the page actually plots | **yes, bundled** |
| `manifest.json` | model revision, hashes, versions, device, seed, date — so "re-derivable" is checkable | yes |
| `figure_1.png`, `figure_2.png` | reference renders; the page redraws these in the browser | reference only |

## 0. How to run this

Two modes, set by one variable in the next cell.

| Mode | Model | Items | Cap | Where it finishes |
|---|---|---|---|---|
| `QUICK` | Qwen3-0.6B | 12 eval + 8 held-out | 512 tokens | a laptop CPU, in minutes |
| `FULL` | Qwen3-4B | 160 eval + 24 held-out | 1024 tokens | a 16 GB GPU, in a couple of hours |

`QUICK` exists because most readers will never run `FULL`. It produces a **legible,
correctly-shaped** figure on a laptop — the bars will be noisy and the levels will be lower, but
the ranking is visible and the pipeline is identical. The page ships `FULL`.

The run is **resumable**: `traces.jsonl` is appended as it goes and re-running skips work already
recorded. Delete the file to start over.

```bash
pip install "torch>=2.7" transformers accelerate sympy matplotlib requests reasoning-from-scratch
```

In [ ]:
# ---------------------------------------------------------------------------
# The two knobs. Everything else is derived from these.
# ---------------------------------------------------------------------------
RUN_MODE = "QUICK"        # "QUICK" (laptop, minutes) or "FULL" (GPU, hours)
SEED = 123                # only selects items; decoding is greedy and unseeded

PRESETS = {
    "QUICK": dict(
        model_id="Qwen/Qwen3-0.6B",
        second_model_id=None,          # two-model panel is off in QUICK
        n_eval=12,                     # evaluated items
        n_pool=8,                      # held-out pool the exemplars are harvested from
        few_shot_k=2,
        max_new_tokens=512,
    ),
    "FULL": dict(
        model_id="Qwen/Qwen3-4B",
        second_model_id="Qwen/Qwen3-1.7B",
        n_eval=160,
        n_pool=24,
        few_shot_k=3,
        max_new_tokens=1024,
    ),
}
CFG = PRESETS[RUN_MODE]

# One generation cap for every rung. This page's variable is the PROMPT, so the
# length budget must not also vary -- that is page 6's experiment. The cap is set
# generously so it should rarely bind; the sweep reports the truncation rate, and
# if it is above a few percent the cap is too low and should be raised.
MAX_NEW_TOKENS = CFG["max_new_tokens"]

# Fallback ladder if the model does not fit: 4B -> 1.7B -> 0.6B. All three are
# hybrid Qwen3 checkpoints with the same tokenizer, so token counts stay
# comparable across them.
#   Qwen3-4B    ~8.0 GB bf16
#   Qwen3-1.7B  ~3.4 GB bf16
#   Qwen3-0.6B  ~1.2 GB bf16

print(f"mode={RUN_MODE}  model={CFG['model_id']}  "
      f"items={CFG['n_eval']} (+{CFG['n_pool']} held out)  cap={MAX_NEW_TOKENS}")

In [ ]:
import hashlib
import json
import random
import re
import subprocess
import time
from datetime import datetime, timezone
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# The inherited harness: text-in, text-out, no model dependency.
from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import (
    load_math500_test,
    render_prompt,
    extract_final_candidate,
    grade_answer,
)

OUT_DIR = Path("out/page_05")
OUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

TRACES_PATH = OUT_DIR / "traces.jsonl"
DERIVED_PATH = OUT_DIR / "derived.json"
MANIFEST_PATH = OUT_DIR / "manifest.json"

mean = lambda xs: sum(xs) / max(len(xs), 1)

print("torch", torch.__version__)

## 1. The grading harness we inherit

Grading free-form maths answers is fiddly and easy to get subtly wrong, so we do not write our
own. `extract_final_candidate` pulls the last `\boxed{...}` out of a completion (falling back to
the last number if the model never boxed anything), and `grade_answer` compares it to ground
truth with light LaTeX canonicalisation and a SymPy equivalence check.

Using the book's grader unchanged means **the grading is not ours to defend** — if you dispute a
score, the argument is with a published, separately-tested function rather than with us.

The cell below is a sanity check, not decoration: it asserts the grader behaves as expected on
cases the sweep will actually hit, including the "model wrote a chain but boxed the answer" case
that the CoT rungs depend on.

In [ ]:
CHECKS = [
    # (raw completion, ground truth, expected)
    (r"The answer is \boxed{9}.",                          "9",              True),
    (r"Step 1: 23-20=3. Step 2: 3+6=9. So \boxed{9}.",     "9",              True),   # a chain
    (r"27",                                                 "9",              False),  # a wrong blurt
    (r"\boxed{\frac{1}{2}}",                               "0.5",            True),   # LaTeX vs decimal
    (r"\boxed{1,024}",                                     "1024",           True),   # thousands separator
    (r"\boxed{\left( 3, \frac{\pi}{2} \right)}",           r"\left( 3, \frac{\pi}{2} \right)", True),
    (r"I am not sure.",                                     "9",              False),  # no answer at all
]

for text, truth, expected in CHECKS:
    got = bool(grade_answer(extract_final_candidate(text), truth))
    status = "ok " if got == expected else "FAIL"
    print(f"{status} extract={extract_final_candidate(text)!r:<40} truth={truth!r:<12} -> {got}")
    assert got == expected, f"grader behaved unexpectedly on {text!r}"

print("\nharness behaves as the sweep assumes.")

## 2. The item set

**MATH-500, levels 4 and 5 only.** The full set sits close to a modern maths-tuned model's
ceiling, and a ladder whose every rung scores 90% measures nothing. Filtering to the two hardest
difficulty levels restores dynamic range without leaving the harness — the dataset already
carries a `level` field, and grading is unchanged.

**Two disjoint splits, and the disjointness is load-bearing:**

- the **eval set**, which every rung is scored on;
- a **held-out pool**, which the worked examples for the one-shot and few-shot rungs are
  harvested from (section 7).

If an exemplar problem also appeared in the eval set, the few-shot rungs would be scored partly
on problems whose answers are printed in their own prompt. The split below makes that impossible
by construction.

Same instrument as pages 6 through 8, which is what makes Act 2 one continuous measurement.

In [ ]:
math500 = load_math500_test(local_path=DATA_DIR / "math500_test.json")

hard = [r for r in math500 if r["level"] in (4, 5)]
hard.sort(key=lambda r: r["unique_id"])          # deterministic before shuffling

rng = random.Random(SEED)
order = list(range(len(hard)))
rng.shuffle(order)

pool_items = [hard[i] for i in order[:CFG["n_pool"]]]
eval_items = [hard[i] for i in order[CFG["n_pool"]:CFG["n_pool"] + CFG["n_eval"]]]

assert not ({r["unique_id"] for r in pool_items} & {r["unique_id"] for r in eval_items}), \
    "exemplar pool leaked into the eval set"


def sha256_of(strings):
    h = hashlib.sha256()
    for s in strings:
        h.update(s.encode("utf-8"))
        h.update(b"\n")
    return h.hexdigest()


ITEM_SET_HASH = sha256_of(r["unique_id"] for r in eval_items)

lv = lambda items, n: sum(r["level"] == n for r in items)
print(f"MATH-500 levels 4+5: {len(hard)} available")
print(f"  eval set    {len(eval_items):>4}  (L4={lv(eval_items,4)}, L5={lv(eval_items,5)})  sha256 {ITEM_SET_HASH[:16]}...")
print(f"  held-out    {len(pool_items):>4}  (L4={lv(pool_items,4)}, L5={lv(pool_items,5)})")
print(f"\nexample item:\n  {eval_items[0]['problem'][:200]}")
print(f"  answer: {eval_items[0]['answer']}")

## 3. The model

Qwen3, loaded through plain `transformers`. Nothing below is specific to the book's
implementation, which is what makes the checkpoint swappable — the fallback ladder in section 0
needs no other edit.

**Precision.** bf16 on GPU, fp32 on CPU. bf16 keeps quantisation out of our own numbers; a
reader re-deriving at 4-bit should see the same *shape* at a slightly different *level*, and that
gap is itself worth knowing about. The dtype actually used is recorded in the manifest.

**Decoding is greedy** — `do_sample=False`, no temperature, no top-p. Sampling is page 7's
subject. Greedy also means the run is reproducible up to floating-point differences between
devices, which is as deterministic as this gets.

In [ ]:
device = get_device()

if device.type == "cpu":
    DTYPE = torch.float32          # bf16 on CPU is slow and patchily supported
else:
    DTYPE = torch.bfloat16

t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(CFG["model_id"])
model = AutoModelForCausalLM.from_pretrained(
    CFG["model_id"], dtype=DTYPE, device_map=None,
).to(device).eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"loaded {CFG['model_id']}  {n_params/1e9:.2f}B params  {DTYPE}  on {device}  "
      f"({time.time()-t0:.1f}s)")

## 4. Proof that thinking is off

This is the cell to read if you suspect we cheated.

Qwen3's chat template takes an `enable_thinking` flag. With it **on**, the model opens a
`<think>` block and reasons inside it before answering — that is the product feature. With it
**off**, the template closes the think block immediately (`<think>\n\n</think>`), so the model has
nowhere to hide a chain and answers directly.

Every generation in this notebook uses `enable_thinking=False`. The two renderings are printed
side by side below so the difference is visible, and the sweep asserts afterwards that no trace
ever contained thinking content.

**Why this matters for the page.** With thinking on, producing a blurt and a chain would take one
boolean flip and would demonstrate nothing about prompting. With it off, any chain that appears
in a trace got there because of *words we put in the prompt*. That is the whole argument.

In [ ]:
demo_msgs = [{"role": "user", "content": "What is 2+2?"}]

off = tokenizer.apply_chat_template(demo_msgs, tokenize=False,
                                    add_generation_prompt=True, enable_thinking=False)
on = tokenizer.apply_chat_template(demo_msgs, tokenize=False,
                                   add_generation_prompt=True, enable_thinking=True)

print("=== enable_thinking=False  <- every generation in this notebook ===")
print(repr(off))
print("\n=== enable_thinking=True   <- never used here, shown for contrast ===")
print(repr(on))
print("\ndifference:", "the OFF template pre-closes the think block" if off != on
      else "NONE -- this tokenizer has no thinking toggle, see note below")

# If a checkpoint has no toggle at all (a non-hybrid model), the page's constraint
# still holds -- thinking is off because the model has no thinking mode -- but say
# so in the manifest rather than implying we disabled something.
HAS_THINKING_TOGGLE = off != on

## 5. The generation loop

Deliberately small and deliberately visible. This is the only place a model is called.

Two details worth pausing on:

**We store token strings, not just text.** The page animates a chain appearing one token at a
time. Replaying that as a character-by-character typewriter would misrepresent generation on a
page whose whole point is that *tokens* are the unit of compute. So each trace carries
`tokens: list[str]`, decoded incrementally so that `"".join(tokens) == text` exactly — asserted
below. Figure 1's reveal steps through real token boundaries.

**We record whether the cap bound.** `truncated=True` means the model was still writing when it
hit `MAX_NEW_TOKENS`. On this page that is a defect, not a variable: page 5 lets every prompt run
to natural length. The sweep reports the rate.

In [ ]:
def incremental_token_strings(ids):
    '''Decode ids one at a time, but incrementally, so multi-byte characters
    split across tokens are not mangled. Concatenates back to the full text.'''
    pieces, prev = [], ""
    for k in range(1, len(ids) + 1):
        cur = tokenizer.decode(ids[:k], skip_special_tokens=True)
        pieces.append(cur[len(prev):])
        prev = cur
    return pieces


@torch.inference_mode()
def generate(prompt_text, mdl=None, tok=None, max_new_tokens=None):
    mdl = mdl or model
    tok = tok or tokenizer
    max_new_tokens = max_new_tokens or MAX_NEW_TOKENS

    chat = tok.apply_chat_template(
        [{"role": "user", "content": prompt_text}],
        tokenize=False, add_generation_prompt=True,
        enable_thinking=False,                      # <- load-bearing, see section 4
    )
    enc = tok(chat, return_tensors="pt").to(mdl.device)

    out = mdl.generate(
        **enc,
        max_new_tokens=max_new_tokens,
        do_sample=False,                            # greedy: no temperature, no top-p
        pad_token_id=tok.pad_token_id or tok.eos_token_id,
    )
    new_ids = out[0][enc["input_ids"].shape[1]:].tolist()

    text = tok.decode(new_ids, skip_special_tokens=True)
    tokens = incremental_token_strings(new_ids)
    assert "".join(tokens) == text, "token strings do not reconstruct the text"

    return {
        "text": text,
        "tokens": tokens,
        "tokens_generated": len(new_ids),
        "truncated": len(new_ids) >= max_new_tokens,
        "thinking_leaked": "<think>" in text,
    }


smoke = generate("What is 2+2? Reply with just the number.", max_new_tokens=32)
print(f"{smoke['tokens_generated']} tokens, truncated={smoke['truncated']}, "
      f"thinking_leaked={smoke['thinking_leaked']}")
print("tokens:", smoke["tokens"][:12])
print("text:", smoke["text"][:200])

## 6. The ladder

Five formulations, from least to most instruction. This is the page's second figure and the
answer to its title: *how to reason* means *what to put in the prompt*.

| Rung | What is added | Whose mechanism |
|---|---|---|
| `bare` | nothing — the harness prompt alone | the true default (ch03's baseline) |
| `forced_blurt` | "Give only the final answer. Do not explain." | the floor |
| `zero_shot_cot` | `"\n\nExplain step by step."` | Kojima et al. 2022 · ch04 §4.3 |
| `one_shot` | one worked example prepended | Wei et al. 2022 |
| `few_shot` | k worked examples prepended | Wei et al. 2022 |

### One thing held constant, on purpose

Every rung — including `bare` — carries ch03's `render_prompt`, which asks for the result as
`\boxed{ANSWER}`. That instruction is about **output format**, not about reasoning, and it has to
be constant or the ladder is confounded: a `bare` rung with no format instruction would score
lower partly because the grader could not find its answer, which would be an artifact of our
measurement rather than a fact about prompting.

So the constant is *how to report*, and the variable is *whether and how to reason*. `bare` here
means "the book's ordinary chapter-3 prompt", which is the honest baseline — it is what you get
if you simply ask.

### Why the few-shot rungs get no "explain step by step"

These are two different techniques and the page cites them separately. The exemplar *is* the
instruction in Wei et al.; the phrase *is* the instruction in Kojima et al. Combining them would
measure a third thing and muddy the ranking.

### Why the floor rung stays

It is tempting to drop `forced_blurt` for tidiness. It should stay: it establishes the floor, and
the gap between it and `bare` shows whether the model was already doing a little reasoning when
nobody asked. On a 2026 instruction-tuned model that gap is often the interesting one — see
section 8.

In [ ]:
RUNGS = [
    ("bare",          "bare question"),
    ("forced_blurt",  "\"answer with the number\""),
    ("zero_shot_cot", "\"explain step by step\""),
    ("one_shot",      "one worked example"),
    ("few_shot",      f"{CFG['few_shot_k']} worked examples"),
]
RUNG_KEYS = [k for k, _ in RUNGS]
RUNG_LABEL = dict(RUNGS)

FORCED_BLURT_SUFFIX = "\n\nGive only the final answer. Do not explain."
COT_SUFFIX = "\n\nExplain step by step."      # ch04 section 4.3, verbatim mechanism


def build_prompt(rung, question, exemplars=None):
    '''The whole ladder, in one readable function. Nothing else varies.'''
    base = render_prompt(question)             # ch03's template -- CONSTANT across rungs

    if rung == "bare":
        return base
    if rung == "forced_blurt":
        return base + FORCED_BLURT_SUFFIX
    if rung == "zero_shot_cot":
        return base + COT_SUFFIX

    k = 1 if rung == "one_shot" else CFG["few_shot_k"]
    shots = "".join(
        render_prompt(e["question"]) + "\n" + e["trace"].strip() + "\n\n"
        for e in (exemplars or [])[:k]
    )
    return shots + base


# The diff that figure 1 highlights: bare -> zero-shot CoT is exactly these words.
print(f"the {len(COT_SUFFIX.split())} words the page highlights:", repr(COT_SUFFIX.strip()))
print()
print("--- bare " + "-" * 60)
print(build_prompt("bare", "What is 23 - 20 + 6?"))
print("\n--- zero_shot_cot " + "-" * 52)
print(build_prompt("zero_shot_cot", "What is 23 - 20 + 6?"))

## 7. The sweep

One pass per rung over the eval set: five passes total. Smaller than page 6's sweep, because the
prompt has five settings and a token budget has seven.

Recorded per item, following the trace-bank schema shared with pages 6 through 9 (`rung` takes
the place of page 6's `budget`):

```
problem_id, level, subject, question, ground_truth,
rung, prompt,            # the exact text sent, before the chat template
trace, tokens,           # full generated text, and its token strings
truncated,
extracted_answer, correct,
tokens_generated,
exemplar_ids,            # which held-out problems were prepended, if any
run_id
```

Full provenance lives once in `manifest.json` rather than being copied onto every line.

The sweep appends to `traces.jsonl` as it goes and skips `(rung, problem_id)` pairs already
present, so an interrupted `FULL` run resumes where it stopped.

In [ ]:
RUN_ID = f"{RUN_MODE.lower()}-{CFG['model_id'].split('/')[-1]}-{ITEM_SET_HASH[:8]}"


def load_done():
    '''(rung, problem_id) -> record, for resume.'''
    done = {}
    if TRACES_PATH.exists():
        with TRACES_PATH.open(encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                r = json.loads(line)
                if r.get("run_id") == RUN_ID:
                    done[(r["rung"], r["problem_id"])] = r
    return done


DONE = load_done()
print(f"run_id={RUN_ID}   {len(DONE)} generations already on disk")


def append_trace(rec):
    with TRACES_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")


def run_rung(rung, items, exemplars=None, mdl=None, tok=None, label=None, split="eval"):
    records, t_start = [], time.time()
    exemplar_ids = [e["problem_id"] for e in (exemplars or [])][
        : (1 if rung == "one_shot" else CFG["few_shot_k"])
    ] if rung in ("one_shot", "few_shot") else []

    for i, row in enumerate(items, start=1):
        key = (rung, row["unique_id"])
        if key in DONE:
            records.append(DONE[key])
            continue

        prompt = build_prompt(rung, row["problem"], exemplars)
        g = generate(prompt, mdl=mdl, tok=tok)
        extracted = extract_final_candidate(g["text"])

        rec = {
            "run_id": RUN_ID,
            "problem_id": row["unique_id"],
            "split": split,                    # "eval" is scored; "held_out" only feeds exemplars
            "level": row["level"],
            "subject": row["subject"],
            "question": row["problem"],
            "ground_truth": row["answer"],
            "rung": rung,
            "prompt": prompt,
            "trace": g["text"],
            "tokens": g["tokens"],
            "truncated": g["truncated"],
            "extracted_answer": extracted,
            "correct": bool(grade_answer(extracted, row["answer"])),
            "tokens_generated": g["tokens_generated"],
            "exemplar_ids": exemplar_ids,
            "thinking_leaked": g["thinking_leaked"],
        }
        append_trace(rec)
        DONE[key] = rec
        records.append(rec)

        elapsed = time.time() - t_start
        eta = elapsed / i * (len(items) - i)
        print(f"\r{label or rung}: {i}/{len(items)}  "
              f"acc={sum(r['correct'] for r in records)/len(records):.2f}  "
              f"eta {eta/60:.1f}m   ", end="")

    print()
    return records

## 8. Pilot: does the bare rung already reason?

Before the full sweep, a short pilot answers the one question that could change the page's
framing.

A 2026 instruction-tuned model may write working when asked a bare maths question, simply because
that is how it was trained to answer. If so, the `bare` rung will not blurt cleanly, the gap
between it and the CoT rung will be small, and figure 1's specimen pair has to come from
`forced_blurt` instead.

**This changes the page's verb, so it is decided from data rather than assumed:**

- *bare blurts* → the page says **"make it reason"**, and figure 1 pairs `bare` against
  `zero_shot_cot`.
- *bare already reasons* → the page says **"let it reason"**, figure 1 pairs `forced_blurt`
  against `zero_shot_cot`, and the caption says plainly that the model's default is already to
  show working. That is a more accurate description of a modern model and it costs the page
  nothing.

The verdict is written into `derived.json` so the page cannot quietly paper over it.

In [ ]:
PILOT_N = min(8, len(eval_items))
pilot_items = eval_items[:PILOT_N]

pilot = {}
for rung in ("bare", "forced_blurt", "zero_shot_cot"):
    pilot[rung] = run_rung(rung, pilot_items, label=f"pilot/{rung}")

def shows_working(rec):
    '''A cheap, stated heuristic: more than one line of substance, or long enough
    that it cannot be a bare assertion.'''
    body = rec["trace"].strip()
    lines = [ln for ln in body.splitlines() if ln.strip()]
    return len(lines) > 2 or rec["tokens_generated"] > 60


pilot_stats = {
    r: dict(
        accuracy=mean([x["correct"] for x in pilot[r]]),
        mean_tokens=mean([x["tokens_generated"] for x in pilot[r]]),
        share_showing_working=mean([shows_working(x) for x in pilot[r]]),
    )
    for r in pilot
}
for r, s in pilot_stats.items():
    print(f"{r:<14} acc={s['accuracy']:.2f}  mean_tokens={s['mean_tokens']:6.1f}  "
          f"shows_working={s['share_showing_working']:.0%}")

BARE_ALREADY_REASONS = pilot_stats["bare"]["share_showing_working"] >= 0.5
FRAMING_VERB = "let it reason" if BARE_ALREADY_REASONS else "make it reason"
FIGURE1_LEFT = "forced_blurt" if BARE_ALREADY_REASONS else "bare"

print(f"\nVERDICT: bare rung {'already reasons' if BARE_ALREADY_REASONS else 'blurts'}")
print(f"  page framing : \"{FRAMING_VERB}\"")
print(f"  figure 1 pairs: {FIGURE1_LEFT}  vs  zero_shot_cot")

## 9. The full sweep, phase A

The three rungs that need no exemplars. `zero_shot_cot` runs on the held-out pool as well —
that pass is what the worked examples are harvested from in the next section.

In [ ]:
results = {}
for rung in ("bare", "forced_blurt", "zero_shot_cot"):
    results[rung] = run_rung(rung, eval_items, label=rung)

# The same prompt, on the held-out pool. These traces never get scored as results;
# their only job is to become exemplars.
pool_cot = run_rung("zero_shot_cot", pool_items, label="pool/zero_shot_cot", split="held_out")

## 10. The worked examples are the model's own

This is the detail that keeps the page honest.

The obvious way to build a few-shot prompt is to write two or three tidy worked solutions
ourselves. If we did that, the page would be measuring *our* prose, and the exemplars would be the
one thing on the page that a reader could not re-derive.

Instead: take **held-out** problems the model already solved **correctly** under the zero-shot CoT
rung, verify them with the same grader used everywhere else, and prepend those. The few-shot
prompt is then built entirely out of the model's own verified-correct work on problems it will
never be scored on. Nothing on the page is written by us.

Selection, in order:

1. correct under the grader,
2. not truncated (a chain cut mid-sentence is a bad example to imitate),
3. short enough that three of them still fit comfortably in context and that the "see full prompt"
   disclosure on the page is readable,
4. shortest first, ties broken by `problem_id` so the choice is deterministic.

In [ ]:
MAX_EXEMPLAR_TOKENS = 400

candidates = [
    r for r in pool_cot
    if r["correct"] and not r["truncated"] and r["tokens_generated"] <= MAX_EXEMPLAR_TOKENS
]
candidates.sort(key=lambda r: (r["tokens_generated"], r["problem_id"]))

exemplars = [
    {
        "problem_id": r["problem_id"],
        "level": r["level"],
        "question": r["question"],
        "trace": r["trace"],
        "ground_truth": r["ground_truth"],
        "extracted_answer": r["extracted_answer"],
        "tokens_generated": r["tokens_generated"],
    }
    for r in candidates[: CFG["few_shot_k"]]
]

if len(exemplars) < CFG["few_shot_k"]:
    print(f"WARNING: only {len(exemplars)} usable exemplars from a pool of {len(pool_items)}.\n"
          f"         Raise n_pool, or MAX_EXEMPLAR_TOKENS, and re-run. The few-shot rung\n"
          f"         below will use however many were found, which weakens the comparison.")

print(f"{len(candidates)} of {len(pool_items)} held-out problems were solved correctly, "
      f"untruncated, and short enough.\n")
for e in exemplars:
    print(f"  {e['problem_id']:<34} L{e['level']}  {e['tokens_generated']:>4} tokens  "
          f"-> {e['extracted_answer']}")

print("\n--- the first exemplar, exactly as it will be prepended " + "-" * 20)
print(render_prompt(exemplars[0]["question"]))
print(exemplars[0]["trace"].strip()[:800])

## 11. The full sweep, phase B

The two rungs that use the harvested exemplars. Note that the prompt for these rungs contains no
instruction to reason — only examples of reasoning.

In [ ]:
for rung in ("one_shot", "few_shot"):
    results[rung] = run_rung(rung, eval_items, exemplars=exemplars, label=rung)

print("\n--- the few-shot prompt, first 1200 characters " + "-" * 26)
print(build_prompt("few_shot", eval_items[0]["problem"], exemplars)[:1200])

## 12. Sweep integrity

Three assertions the page's honesty rests on. If any of these fails, the numbers below do not mean
what the page says they mean.

In [ ]:
all_records = [r for rung in RUNG_KEYS for r in results[rung]]

leaked = [r for r in all_records if r["thinking_leaked"]]
print(f"1. thinking never leaked into a trace ......... {len(leaked)} violations")
assert not leaked, "a trace contained a <think> block; the page's core constraint is broken"

trunc_rate = sum(r["truncated"] for r in all_records) / len(all_records)
print(f"2. the generation cap rarely bound ............ {trunc_rate:.1%} truncated")
if trunc_rate > 0.05:
    print("   ^ above 5%: raise MAX_NEW_TOKENS and re-run. Page 5 lets every prompt run to")
    print("     natural length; capping it is page 6's experiment, not this one.")

ex_ids = {e["problem_id"] for e in exemplars}
overlap = ex_ids & {r["problem_id"] for r in results["few_shot"]}
print(f"3. exemplars stayed out of the eval set ....... {len(overlap)} overlaps")
assert not overlap, "an exemplar problem was also scored"

print(f"\n{len(all_records)} generations across {len(RUNG_KEYS)} rungs and "
      f"{len(eval_items)} items.")

## 13. Beat 1 — the lever

*Figure 1 on the page.* Two real completions, side by side, from one model at one setting. The
only difference between the two prompts is the highlighted words.

The specimen is **curated from the sweep, not generated for the figure** — we search the run for a
problem where the left-hand rung gets it wrong with a short answer and the CoT rung gets it right
with a chain short enough to walk on screen. That pairing is common; legibility, not existence, is
the binding constraint, so the search sorts by chain length.

Which rung sits on the left was decided by the pilot in section 8.

In [ ]:
MAX_SPECIMEN_TOKENS = 260      # a chain the reader can walk on screen

by_id = {r["problem_id"]: r for r in results[FIGURE1_LEFT]}
pairs = []
for right in results["zero_shot_cot"]:
    left = by_id.get(right["problem_id"])
    if not left:
        continue
    if (not left["correct"]) and right["correct"] and not right["truncated"]:
        pairs.append((left, right))

pairs.sort(key=lambda p: (p[1]["tokens_generated"], p[1]["problem_id"]))
print(f"{len(pairs)} problems where {FIGURE1_LEFT} misses and zero_shot_cot lands.")

if not pairs:
    raise RuntimeError(
        "No specimen pair found. In QUICK mode with 12 items this can happen by chance -- "
        "raise n_eval, or run FULL. Do not hand-pick or author a pair."
    )

left_spec, right_spec = next(
    (p for p in pairs if p[1]["tokens_generated"] <= MAX_SPECIMEN_TOKENS), pairs[0]
)

print(f"\nchosen: {right_spec['problem_id']} (L{right_spec['level']}, "
      f"chain = {right_spec['tokens_generated']} tokens)\n")
print("QUESTION:", right_spec["question"][:300])
print("TRUTH   :", right_spec["ground_truth"])
print(f"\n--- {FIGURE1_LEFT}: {left_spec['tokens_generated']} tokens, "
      f"answered {left_spec['extracted_answer']!r}  [WRONG] ---")
print(left_spec["trace"].strip()[:600])
print(f"\n--- zero_shot_cot: {right_spec['tokens_generated']} tokens, "
      f"answered {right_spec['extracted_answer']!r}  [RIGHT] ---")
print(right_spec["trace"].strip()[:1200])

In [ ]:
# The prompt diff the page highlights. On the page this is rendered as one prompt
# with the added words picked out in colour -- the whole argument is in that highlight.
left_prompt = left_spec["prompt"]
right_prompt = right_spec["prompt"]

assert right_prompt.startswith(left_prompt.replace(FORCED_BLURT_SUFFIX, "")), \
    "the two prompts do not share a common stem; the figure's premise fails"

shared_stem = left_prompt.replace(FORCED_BLURT_SUFFIX, "")
PROMPT_DIFF = {
    "shared": shared_stem,
    "left_added": left_prompt[len(shared_stem):],
    "right_added": right_prompt[len(shared_stem):],
}
print("shared stem, last 120 chars:", repr(shared_stem[-120:]))
print("left adds :", repr(PROMPT_DIFF["left_added"]))
print("right adds:", repr(PROMPT_DIFF["right_added"]))
print(f"\n-> {len(PROMPT_DIFF['right_added'].split())} words separate a "
      f"{left_spec['tokens_generated']}-token miss from a "
      f"{right_spec['tokens_generated']}-token hit.")

## 14. Beat 2 — the ladder, measured

*Figure 2 on the page*, replacing the emergence chart entirely.

Horizontal bars, one per rung, same model, same items, same decoding, thinking off throughout.
**Model size appears on no axis** — that is the structural guarantee that this page cannot be read
as making a claim about scale, which is a different subject with a different evidentiary bar.

The cost row underneath is the handoff to page 6.

In [ ]:
def summarise(rung):
    rs = results[rung]
    toks = sorted(r["tokens_generated"] for r in rs)
    return {
        "rung": rung,
        "label": RUNG_LABEL[rung],
        "n": len(rs),
        "n_correct": sum(r["correct"] for r in rs),
        "accuracy": mean([r["correct"] for r in rs]),
        "mean_tokens": mean(toks),
        "median_tokens": toks[len(toks) // 2],
        "total_tokens": sum(toks),
        "truncated": sum(r["truncated"] for r in rs),
    }


summary = [summarise(k) for k in RUNG_KEYS]

print(f"{'rung':<16}{'prompt':<26}{'solved':>10}{'accuracy':>10}{'mean tok':>10}")
print("-" * 72)
for s in summary:
    print(f"{s['rung']:<16}{s['label']:<26}{s['n_correct']:>4}/{s['n']:<5}"
          f"{s['accuracy']:>9.1%}{s['mean_tokens']:>10.0f}")

best = max(summary, key=lambda s: s["accuracy"])
floor = min(summary, key=lambda s: s["accuracy"])
print(f"\nbest rung : {best['rung']} at {best['accuracy']:.1%}, "
      f"{best['mean_tokens']:.0f} tokens per problem")
print(f"floor rung: {floor['rung']} at {floor['accuracy']:.1%}, "
      f"{floor['mean_tokens']:.0f} tokens per problem")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 3.6))
ys = range(len(summary))[::-1]
ax.barh(list(ys), [s["accuracy"] * 100 for s in summary], height=0.55, color="#3b4a6b")
ax.set_yticks(list(ys))
ax.set_yticklabels([s["label"] for s in summary])
ax.set_xlabel("share of problems solved (%)")
ax.set_xlim(0, 100)
ax.spines[["top", "right"]].set_visible(False)

for y, s in zip(ys, summary):
    ax.text(s["accuracy"] * 100 + 1.2, y, f"{s['accuracy']*100:.0f}%",
            va="center", fontsize=9)

cost = "  ·  ".join(f"{s['mean_tokens']:.0f}" for s in summary)
ax.set_title(f"{CFG['model_id']} · thinking off · greedy · MATH-500 L4-5, n={len(eval_items)}",
             fontsize=10, loc="left")
fig.text(0.01, -0.04, f"tokens written, same order:  {cost}", fontsize=9, color="#555")
fig.tight_layout()
fig.savefig(OUT_DIR / "figure_2.png", dpi=160, bbox_inches="tight")
plt.show()

### Reading the ladder

Write down what the run actually produced before deciding what the page says about it. The
questions the caption has to answer:

- **Does a worked example beat the magic phrase?** This is the genuinely open question and the
  reason the figure exists. Readers wonder about it and nothing on the current page answers it.
- **Do `one_shot` and `few_shot` land on top of each other?** If so, collapse them into one rung
  and reclaim the space — three exemplars was a guess.
- **How far above the floor is `bare`?** That gap is how much reasoning the model does when
  nobody asked, and it is what section 8's framing decision turns on.
- **Is anything non-monotone?** If a rung is worse than the one below it, keep it and show it.
  A surprise in the ranking is the best thing that can happen to this figure, and the caption
  should name it rather than smooth it away.

In [ ]:
# Written out so the caption is drafted against numbers rather than expectations.
acc = {s["rung"]: s["accuracy"] for s in summary}
tok = {s["rung"]: s["mean_tokens"] for s in summary}

print(f"bare over the floor      : {(acc['bare']-acc['forced_blurt'])*100:+.1f} pts  "
      f"({tok['bare']:.0f} vs {tok['forced_blurt']:.0f} tokens)")
print(f"the phrase over bare     : {(acc['zero_shot_cot']-acc['bare'])*100:+.1f} pts  "
      f"({tok['zero_shot_cot']:.0f} vs {tok['bare']:.0f} tokens)")
print(f"one example over phrase  : {(acc['one_shot']-acc['zero_shot_cot'])*100:+.1f} pts")
print(f"{CFG['few_shot_k']} examples over one       : {(acc['few_shot']-acc['one_shot'])*100:+.1f} pts")

ordered = [s["rung"] for s in sorted(summary, key=lambda s: s["accuracy"])]
print(f"\nranking, worst to best: {' < '.join(ordered)}")
expected = ["forced_blurt", "bare", "zero_shot_cot", "one_shot", "few_shot"]
print("matches the naive expectation" if ordered == expected
      else f"DOES NOT match the naive expectation {' < '.join(expected)} -- the caption should say so")

collapse = abs(acc["few_shot"] - acc["one_shot"]) * 100 < 2.0
print(f"\none_shot and few_shot within 2 points: {collapse}"
      f"{'  -> consider collapsing them into one rung' if collapse else ''}")

## 15. Beat 3 — the close

Not a third chart: one paired readout. The best rung's accuracy next to what it cost to get
there.

The sentence the page ends on is that **the chain was not free**. The reader has just watched the
token count go from single digits to hundreds for a few points of accuracy, so page 6's opening
question — *how much of that length do you actually need?* — is no longer abstract. Page 5 varies
the prompt at unlimited length; page 6 fixes the prompt and varies the length. Same model, same
items, same harness: one experiment read twice.

In [ ]:
close = {
    "best_rung": best["rung"],
    "best_label": best["label"],
    "best_accuracy": best["accuracy"],
    "best_mean_tokens": best["mean_tokens"],
    "floor_rung": floor["rung"],
    "floor_label": floor["label"],
    "floor_accuracy": floor["accuracy"],
    "floor_mean_tokens": floor["mean_tokens"],
    "accuracy_gain_pts": (best["accuracy"] - floor["accuracy"]) * 100,
    "token_multiple": best["mean_tokens"] / max(floor["mean_tokens"], 1e-9),
}
print(f"{close['best_label']:<26} {close['best_accuracy']:>6.1%}   "
      f"{close['best_mean_tokens']:>6.0f} tokens/problem")
print(f"{close['floor_label']:<26} {close['floor_accuracy']:>6.1%}   "
      f"{close['floor_mean_tokens']:>6.0f} tokens/problem")
print(f"\n{close['accuracy_gain_pts']:+.1f} points of accuracy, "
      f"for {close['token_multiple']:.0f}x the tokens.")

## 16. Optional — the two-model panel

*Off by default.* This is the one element on the page that could be misread as a claim about
model size, so it is built deliberately and cut without regret.

**The strict rule.** The comparison is about **whether a chain appears and what shape it takes**,
never about who got the answer right. The finding is that *the prompt works on both* — both models
produce a chain, and the chains differ in quality, which is what the rest of Act 2 is about.

Accuracy is deliberately not computed here. If the panel cannot be captioned without implying that
the smaller model cannot reason, **cut the panel**. Page 5 makes no claim about size, and no
figure on it puts size on an axis.

In [ ]:
RUN_TWO_MODEL_PANEL = False       # set True only if you intend to caption it strictly

two_model_panel = None
if RUN_TWO_MODEL_PANEL and CFG["second_model_id"]:
    tok2 = AutoTokenizer.from_pretrained(CFG["second_model_id"])
    mdl2 = AutoModelForCausalLM.from_pretrained(
        CFG["second_model_id"], dtype=DTYPE).to(device).eval()

    prompt = build_prompt("zero_shot_cot", right_spec["question"])
    g2 = generate(prompt, mdl=mdl2, tok=tok2)

    shape = lambda text, n: {
        "tokens_generated": n,
        "lines": len([l for l in text.splitlines() if l.strip()]),
        "has_chain": len([l for l in text.splitlines() if l.strip()]) > 2,
    }
    two_model_panel = {
        "problem_id": right_spec["problem_id"],
        "prompt_rung": "zero_shot_cot",
        "models": [
            {"model_id": CFG["model_id"], "trace": right_spec["trace"],
             "tokens": right_spec["tokens"],
             **shape(right_spec["trace"], right_spec["tokens_generated"])},
            {"model_id": CFG["second_model_id"], "trace": g2["text"],
             "tokens": g2["tokens"], **shape(g2["text"], g2["tokens_generated"])},
        ],
        "caption_rule": ("Chain shape only. Both models produce a chain because the prompt works "
                        "on both. Never compare correctness here; page 5 makes no size claim."),
    }
    for m in two_model_panel["models"]:
        print(f"{m['model_id']:<20} chain={m['has_chain']}  "
              f"{m['lines']} lines  {m['tokens_generated']} tokens")
    del mdl2, tok2
else:
    print("two-model panel skipped.")

## 17. Artifacts

`derived.json` is the contract between this notebook and the page. The page reads it and nothing
else; **no inference runs at demonstration time.**

It carries the aggregates the page plots, the specimens figure 1 displays (with their token
strings, so the reveal animation steps through real token boundaries), the harvested exemplars
behind the "see full prompt" disclosure, and the framing verdict from section 8 — so a decision
made from data cannot later be quietly reversed in prose.

`traces.jsonl` stays out of the web bundle and is published as a download beside the notebook
link. The page ships what it draws and links what it drew it from.

In [ ]:
def git_sha():
    try:
        return subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True,
                              text=True, check=True).stdout.strip()
    except Exception:
        return None


def repo_revision(model_id):
    try:
        from huggingface_hub import model_info
        return model_info(model_id).sha
    except Exception:
        return None


def tokenizer_fingerprint(tok):
    try:
        return hashlib.sha256(tok.backend_tokenizer.to_str().encode()).hexdigest()
    except Exception:
        return None


manifest = {
    "run_id": RUN_ID,
    "notebook": "page_05_how_to_reason.ipynb",
    "page": 5,
    "run_mode": RUN_MODE,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "notebook_git_sha": git_sha(),
    "model": {
        "id": CFG["model_id"],
        "revision": repo_revision(CFG["model_id"]),
        "tokenizer_sha256": tokenizer_fingerprint(tokenizer),
        "params": n_params,
        "dtype": str(DTYPE),
        "thinking": "off",
        "has_thinking_toggle": HAS_THINKING_TOGGLE,
    },
    "decoding": {"strategy": "greedy", "do_sample": False,
                 "max_new_tokens": MAX_NEW_TOKENS},
    "items": {
        "source": "MATH-500 (ch03 copy)",
        "filter": "level in {4, 5}",
        "n_eval": len(eval_items),
        "n_held_out": len(pool_items),
        "eval_set_sha256": ITEM_SET_HASH,
        "seed": SEED,
    },
    "grading": {"extract": "reasoning_from_scratch.ch03.extract_final_candidate",
                "grade": "reasoning_from_scratch.ch03.grade_answer"},
    "chat_template_thinking_off": off,
    "versions": {"torch": torch.__version__,
                 "transformers": __import__("transformers").__version__,
                 "device": str(device)},
    "integrity": {"thinking_leaks": len(leaked), "truncation_rate": trunc_rate},
}

derived = {
    "manifest": manifest,
    "framing": {
        "bare_rung_already_reasons": BARE_ALREADY_REASONS,
        "verb": FRAMING_VERB,
        "figure1_left_rung": FIGURE1_LEFT,
        "pilot": pilot_stats,
        "note": ("Decided from the pilot in section 8, not assumed. If the bare rung already "
                 "shows working, the page says 'let it reason' and figure 1 pairs the forced "
                 "blurt against the chain."),
    },
    "figure_1": {
        "left": {k: left_spec[k] for k in
                 ("rung", "trace", "tokens", "tokens_generated", "extracted_answer", "correct")},
        "right": {k: right_spec[k] for k in
                  ("rung", "trace", "tokens", "tokens_generated", "extracted_answer", "correct")},
        "problem_id": right_spec["problem_id"],
        "level": right_spec["level"],
        "question": right_spec["question"],
        "ground_truth": right_spec["ground_truth"],
        "prompt_diff": PROMPT_DIFF,
        "n_candidate_pairs": len(pairs),
    },
    "figure_2": {
        "rungs": summary,
        "ranking_worst_to_best": ordered,
        "matches_naive_expectation": ordered == expected,
        "one_shot_few_shot_within_2pts": collapse,
    },
    "close": close,
    "exemplars": exemplars,
    "two_model_panel": two_model_panel,
}

DERIVED_PATH.write_text(json.dumps(derived, indent=2, ensure_ascii=False), encoding="utf-8")
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")

kb = lambda p: p.stat().st_size / 1024
print(f"derived.json   {kb(DERIVED_PATH):>9.1f} KB   <- bundled with the page")
print(f"manifest.json  {kb(MANIFEST_PATH):>9.1f} KB   <- bundled with the page")
print(f"traces.jsonl   {kb(TRACES_PATH):>9.1f} KB   <- published as a download")

## 18. What the page is allowed to claim

Written down so the translation step in `docs/page_6_refactor.md` §6 has something to check
itself against.

**Supported by this run:**

- Adding `"Explain step by step."` to the prompt changes what the model writes, on a specific
  model at a specific setting, with both traces shown in full and re-derivable from this file.
- The five formulations rank as `figure_2.ranking_worst_to_best` reports, on MATH-500 levels 4
  and 5, at *n* = the eval-set size, with the model's thinking mode off throughout.
- The formulation that scores highest also writes the most tokens.
- The worked examples in the few-shot prompts are the model's own verified-correct solutions to
  held-out problems.

**Not supported, and not to appear on the page:**

- Anything about **model size**. One checkpoint carries the ladder; the optional panel compares
  chain *shape* on two, and nothing on this page puts size on an axis.
- Anything about **emergence**. Where prompting stops working is a different subject with a
  different evidentiary bar, and two checkpoints cannot meet it.
- Any **general** claim about chain-of-thought. This is one model, one dataset, one difficulty
  band, greedy. `QUICK` mode especially is a shape check, not an estimate.
- Any number attributed to **Wei et al.** or **Kojima et al.** The citations on the page are for
  the *techniques* those papers introduced. Every number here is ours, and no cite sits where it
  could imply otherwise.

**Known limitations, stated rather than buried:**

- *n* is small — 160 items in `FULL` — so differences of a couple of points between adjacent rungs
  are not resolvable. The caption should not rank rungs that overlap.
- Greedy decoding on different hardware can diverge on a handful of items through floating-point
  differences. Re-deriving should reproduce the shape and the ranking, not every digit.
- The exemplar harvest picks the *shortest* correct held-out solutions, which is a legibility
  choice with a cost: short solutions may be easier problems, so the exemplars are not a random
  sample of the model's correct work.
- `forced_blurt` is one phrasing of "do not explain". A different phrasing would move the floor.

---

### Next steps in the pipeline

Per `docs/page_6_refactor.md` §6, which is canonical for every page resting on traces:

| # | Step | State |
|---|---|---|
| 1 | Spec implemented as a notebook | **this file** |
| 2 | Refined: hardware-agnostic, beginner-readable | QUICK/FULL modes, fallback ladder, plain transformers |
| 3 | Notebook approved | ← human + AI sign-off next |
| 4 | Translated to JS; page renders committed logs | after (3) |
| 5 | Post-translation formatting and balance | |

Step 4 reimplements the **derivation** logic in TypeScript, not the inference: the aggregates in
`figure_2.rungs` and `close` get TS equivalents, and a cross-language test recomputes them from
`traces.jsonl` and asserts they match `derived.json`. That test is where translation bugs would
otherwise hide.